In [1]:
# ==================================================================================
# FULL CODE: DUAL-STREAM FASTER R-CNN (FIXED NAN & ANCHOR ERRORS)
# PLATFORM: KAGGLE (P100/T4 GPU SUPPORTED)
# ==================================================================================

import os
import cv2
import gc
import numpy as np
import pandas as pd
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign, box_iou, FeaturePyramidNetwork
from torchvision.transforms import v2 as T
from torchvision import tv_tensors 
from tqdm.auto import tqdm
from collections import OrderedDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# ==========================================
# 1. CẤU HÌNH TỐI ƯU (CONFIGURATION)
# ==========================================
class Config:
    SEED = 42
    DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    
    # --- FIX CRASH DATALOADER ---
    # Giảm xuống 1 hoặc 0 để tránh lỗi "can only test a child process" khi train bị fail
    NUM_WORKERS = 1 
    
    # --- OPTIMIZATION PARAMETERS ---
    IMG_SIZE = 512          
    BATCH_SIZE = 4          
    REAL_BATCH_SIZE = 2     
    
    NUM_CLASSES = 2          
    NUM_EPOCHS = 15         
    
    # --- FIX NAN ERROR: LEARNING RATE ---
    # Giảm LR xuống mức an toàn cho kiến trúc Attention phức tạp
    LR = 2e-5               
    MIN_LR = 1e-7
    WEIGHT_DECAY = 0.01     
    USE_AMP = True          
    
    TRAIN_DIR = '/kaggle/input/rsna-pneumonia-processed-dataset/Training/Images'
    CSV_FILE = '/kaggle/input/rsna-pneumonia-processed-dataset/stage2_train_metadata.csv'

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(Config.SEED)

# ==========================================
# 2. DATASET & TRANSFORMS
# ==========================================
def get_transforms():
    train_geo = T.Compose([
        T.Resize((Config.IMG_SIZE, Config.IMG_SIZE), antialias=True),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomAffine(degrees=10, translate=(0.15, 0.15), scale=(0.9, 1.1)), 
        T.ClampBoundingBoxes() 
    ])
    
    val_geo = T.Compose([
        T.Resize((Config.IMG_SIZE, Config.IMG_SIZE), antialias=True),
        T.ClampBoundingBoxes()
    ])

    s1_pixel = T.Compose([
        T.ColorJitter(brightness=0.15, contrast=0.15),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    s2_pixel = T.Compose([
        T.RandomEqualize(p=1.0), 
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    return (train_geo, s1_pixel, s2_pixel), (val_geo, s1_pixel, s2_pixel)

class RSNADataset(Dataset):
    def __init__(self, dataframe, image_dir, transforms_set=None):
        self.df = dataframe
        self.image_dir = image_dir
        if transforms_set:
            self.geo, self.s1, self.s2 = transforms_set
        else:
            self.geo, self.s1, self.s2 = None, None, None

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, f"{row['patientId']}.png")
        
        try:
            img = cv2.imread(img_path)
            if img is None: raise ValueError
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        except:
            img = np.zeros((Config.IMG_SIZE, Config.IMG_SIZE, 3), dtype=np.uint8)
            
        H, W, _ = img.shape
        boxes, labels = [], []
        
        if row['Target'] == 1:
            raw_boxes = row['boxes'] if isinstance(row['boxes'], list) else []
            for box in raw_boxes:
                x, y, w, h = box
                boxes.append([x, y, x + w, y + h])
                labels.append(1)

        img_t = torch.from_numpy(img).permute(2, 0, 1)
        
        if len(boxes) > 0:
            boxes_t = torch.tensor(boxes, dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.int64)
            boxes_tv = tv_tensors.BoundingBoxes(boxes_t, format="XYXY", canvas_size=(H, W))
        else:
            boxes_tv = tv_tensors.BoundingBoxes(torch.zeros((0, 4), dtype=torch.float32), format="XYXY", canvas_size=(H, W))
            labels_t = torch.zeros((0,), dtype=torch.int64)

        if self.geo:
            img_geo, boxes_geo, labels_geo = self.geo(img_t, boxes_tv, labels_t)
        else:
            img_geo, boxes_geo, labels_geo = img_t, boxes_tv, labels_t

        img_s1 = self.s1(img_geo) 
        img_s2 = self.s2(img_geo)
        
        combined = torch.cat([img_s1, img_s2], dim=0) 

        if boxes_geo.shape[0] > 0:
            keep = (boxes_geo[:, 2] > boxes_geo[:, 0]) & (boxes_geo[:, 3] > boxes_geo[:, 1])
            boxes_geo = boxes_geo[keep]
            labels_geo = labels_geo[keep]

        target = {"boxes": boxes_geo, "labels": labels_geo, "image_id": torch.tensor([idx])}
        return combined, target

def collate_fn(batch):
    images, targets = zip(*batch)
    return torch.stack(images, dim=0), targets


# 3. MODEL ARCHITECTURE (FIXED NAN & SIZES)
class ParallelAttention(nn.Module):
    def __init__(self, dim, num_heads=4, dropout_rate=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.to_q = nn.Linear(dim, dim, bias=False)
        self.to_k = nn.Linear(dim, dim, bias=False)
        self.to_v = nn.Linear(dim, dim, bias=False)
        self.proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x_query, x_key):
        B, C, H, W = x_query.shape
        q = x_query.flatten(2).transpose(1, 2)
        k = x_key.flatten(2).transpose(1, 2)
        v = x_key.flatten(2).transpose(1, 2)
        
        q = self.to_q(q).reshape(B, H*W, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        k = self.to_k(k).reshape(B, H*W, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        v = self.to_v(v).reshape(B, H*W, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.float().softmax(dim=-1).to(q.dtype)
        attn = self.dropout(attn)
        
        out = attn @ v
        out = out.transpose(1, 2).reshape(B, H*W, C)
        out = self.proj(out)
        return out.transpose(1, 2).reshape(B, C, H, W)

class ParallelFusionModule(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.attn = ParallelAttention(in_channels)
        self.reduce = nn.Conv2d(in_channels * 2, in_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.MAX_DIM = 32 

    def forward(self, x1, x2):
        B, C, H, W = x1.shape
        
        if H > self.MAX_DIM or W > self.MAX_DIM:
            x1_s = F.adaptive_avg_pool2d(x1, (self.MAX_DIM, self.MAX_DIM))
            x2_s = F.adaptive_avg_pool2d(x2, (self.MAX_DIM, self.MAX_DIM))
        else:
            x1_s, x2_s = x1, x2
            
        attn_out1 = self.attn(x1_s, x2_s) 
        attn_out2 = self.attn(x2_s, x1_s)
        
        if H > self.MAX_DIM or W > self.MAX_DIM:
            attn_out1 = F.interpolate(attn_out1, size=(H, W), mode='bilinear', align_corners=False)
            attn_out2 = F.interpolate(attn_out2, size=(H, W), mode='bilinear', align_corners=False)
            
        x1_refined = x1 + attn_out1
        x2_refined = x2 + attn_out2
        
        x_fused = torch.cat([x1_refined, x2_refined], dim=1)
        x_fused = self.reduce(x_fused)
        return self.relu(self.bn(x_fused))

class DualStreamBackboneWithFPN(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = torchvision.models.resnet50(weights='IMAGENET1K_V1')
        
        self.conv1, self.bn1, self.relu, self.maxpool = resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool
        self.layer1 = resnet.layer1 # 256
        self.layer2 = resnet.layer2 # 512
        self.layer3 = resnet.layer3 # 1024
        self.layer4 = resnet.layer4 # 2048

        self.fuse1 = ParallelFusionModule(256)
        self.fuse2 = ParallelFusionModule(512)
        self.fuse3 = ParallelFusionModule(1024)
        self.fuse4 = ParallelFusionModule(2048)

        self.fpn = FeaturePyramidNetwork(
            in_channels_list=[256, 512, 1024, 2048],
            out_channels=256
        )

        for module in [self.conv1, self.bn1, self.layer1, self.layer2, self.layer3, self.layer4]:
            for layer in module.modules():
                if isinstance(layer, nn.BatchNorm2d):
                    layer.eval() 
                    for param in layer.parameters(): param.requires_grad = False

    def forward_single_stream(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        return c1, c2, c3, c4

    def forward(self, x):
        x_s1, x_s2 = x[:, :3], x[:, 3:]
        s1_c1, s1_c2, s1_c3, s1_c4 = self.forward_single_stream(x_s1)
        s2_c1, s2_c2, s2_c3, s2_c4 = self.forward_single_stream(x_s2)

        f_c1 = self.fuse1(s1_c1, s2_c1)
        f_c2 = self.fuse2(s1_c2, s2_c2)
        f_c3 = self.fuse3(s1_c3, s2_c3)
        f_c4 = self.fuse4(s1_c4, s2_c4)

        features = OrderedDict([
            ('feat0', f_c1), ('feat1', f_c2), ('feat2', f_c3), ('feat3', f_c4)
        ])
        
        return self.fpn(features)

def get_model():
    backbone = DualStreamBackboneWithFPN()
    backbone.out_channels = 256 

    # --- FIX ANCHOR ASSERTION ERROR ---
    # FPN trả về 4 levels (feat0..3), nên cần khai báo đúng 4 sizes.
    anchor_generator = AnchorGenerator(
        sizes=((32,), (64,), (128,), (256,)), 
        aspect_ratios=((0.5, 0.75, 1.0, 1.5, 2.0),) * 4
    )
    
    roi_pooler = MultiScaleRoIAlign(
        featmap_names=['feat0', 'feat1', 'feat2', 'feat3'], 
        output_size=7, sampling_ratio=2
    )
    
    model = FasterRCNN(
        backbone, num_classes=Config.NUM_CLASSES,
        rpn_anchor_generator=anchor_generator, box_roi_pool=roi_pooler,
        image_mean=[0.0]*6, image_std=[1.0]*6, 
        min_size=Config.IMG_SIZE, max_size=Config.IMG_SIZE,
        rpn_pre_nms_top_n_train=4000, rpn_post_nms_top_n_train=2000,
        rpn_pre_nms_top_n_test=2000, rpn_post_nms_top_n_test=1000,
        rpn_nms_thresh=0.7,
        box_score_thresh=0.05, 
        box_nms_thresh=0.45,
        box_detections_per_img=50
    )
    return model

# ==========================================
# 4. EVALUATION
# ==========================================
def calculate_average_precision(recalls, precisions):
    recalls = torch.cat((torch.tensor([0.0]), recalls, torch.tensor([1.0])))
    precisions = torch.cat((torch.tensor([0.0]), precisions, torch.tensor([0.0])))
    for i in range(len(precisions) - 2, -1, -1):
        precisions[i] = torch.max(precisions[i], precisions[i + 1])
    indices = torch.where(recalls[1:] != recalls[:-1])[0]
    ap = torch.sum((recalls[indices + 1] - recalls[indices]) * precisions[indices + 1])
    return ap.item()

def get_rsna_score(model, loader, device):
    model.eval()
    all_pred_boxes, all_pred_scores, all_pred_ids = [], [], []
    all_gt_boxes, all_gt_ids = [], []
    y_true_cls, y_pred_cls = [], []
    
    print("⏳ Running Evaluation...")
    with torch.no_grad():
        for batch_idx, (images, targets) in enumerate(tqdm(loader, desc="Eval", leave=False)):
            images = images.to(device)
            outputs = model(images)
            
            for i, output in enumerate(outputs):
                image_id = batch_idx * loader.batch_size + i
                scr = output['scores'].cpu()
                box = output['boxes'].cpu()
                
                y_pred_cls.append(scr.max().item() if len(scr) > 0 else 0.0)
                all_pred_boxes.append(box)
                all_pred_scores.append(scr)
                all_pred_ids.extend([image_id] * len(box))
                
                tgt_box = targets[i]['boxes'].cpu()
                y_true_cls.append(1 if len(tgt_box) > 0 else 0)
                all_gt_boxes.append(tgt_box)
                all_gt_ids.extend([image_id] * len(tgt_box))

    try: auc = roc_auc_score(y_true_cls, y_pred_cls)
    except: auc = 0.0

    if len(all_pred_boxes) > 0:
        cat_pred_boxes = torch.cat(all_pred_boxes)
        cat_pred_scores = torch.cat(all_pred_scores)
        cat_pred_ids = torch.tensor(all_pred_ids)
    else:
        cat_pred_boxes = torch.empty((0,4)); cat_pred_scores = torch.empty((0)); cat_pred_ids = torch.empty((0))

    cat_gt_boxes = torch.cat(all_gt_boxes)
    cat_gt_ids = torch.tensor(all_gt_ids)
    
    aps = []
    iou_thresholds = [0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75]
    
    for iou_thresh in iou_thresholds:
        if len(cat_pred_scores) == 0: aps.append(0.0); continue
        
        sorted_idxs = torch.argsort(cat_pred_scores, descending=True)
        p_boxes = cat_pred_boxes[sorted_idxs]; p_ids = cat_pred_ids[sorted_idxs]
        tp = torch.zeros(len(p_boxes)); fp = torch.zeros(len(p_boxes)); gt_used = set()
        
        for idx in range(len(p_boxes)):
            img_id = p_ids[idx].item()
            pred_box = p_boxes[idx].unsqueeze(0)
            mask_gt = (cat_gt_ids == img_id)
            img_gt_boxes = cat_gt_boxes[mask_gt]
            
            if len(img_gt_boxes) == 0: fp[idx] = 1; continue
            
            ious = box_iou(pred_box, img_gt_boxes)
            max_iou, max_idx = ious.max(dim=1)
            
            if max_iou > iou_thresh:
                gt_indices_in_subset = torch.where(mask_gt)[0]
                global_idx = gt_indices_in_subset[max_idx].item()
                if global_idx not in gt_used: tp[idx] = 1; gt_used.add(global_idx)
                else: fp[idx] = 1 
            else: fp[idx] = 1
        
        cum_tp = torch.cumsum(tp, dim=0); cum_fp = torch.cumsum(fp, dim=0)
        recalls = cum_tp / (len(cat_gt_boxes) + 1e-6)
        precisions = cum_tp / (cum_tp + cum_fp + 1e-6)
        aps.append(calculate_average_precision(recalls, precisions))

    mean_ap = np.mean(aps)
    print(f"📊 EVAL: AUC = {auc:.4f} | mAP@{iou_thresholds[0]}-{iou_thresholds[-1]} = {mean_ap:.4f}")
    return mean_ap

# ==========================================
# 5. TRAINING LOOP
# ==========================================
def train_one_epoch(model, optimizer, loader, scaler, accum_iter, epoch):
    model.train()
    loss_total = 0
    optimizer.zero_grad()
    
    pbar = tqdm(loader, desc=f"Train Ep {epoch+1}")
    for i, (images, targets) in enumerate(pbar):
        # Warmup
        if epoch == 0 and i < 500:
            lr_scale = min(1., float(i + 1) / 500.)
            for pg in optimizer.param_groups: pg['lr'] = Config.LR * lr_scale

        images = images.to(Config.DEVICE)
        targets = [{k: v.to(Config.DEVICE) for k, v in t.items()} for t in targets]
        
        with torch.amp.autocast('cuda', enabled=Config.USE_AMP):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values()) / accum_iter
        
        # --- FIX NAN: CHECK LOSS VALUE ---
        if not torch.isfinite(losses):
            print(f"⚠️ Warning: Loss NaN/Inf at step {i}. Skipping.")
            optimizer.zero_grad()
            scaler.update() 
            continue

        scaler.scale(losses).backward()
        
        if (i + 1) % accum_iter == 0:
            scaler.unscale_(optimizer)
            # --- FIX NAN: HARD CLIP ---
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        loss_total += losses.item() * accum_iter
        pbar.set_postfix(loss=losses.item() * accum_iter, lr=optimizer.param_groups[0]['lr'])
        
    return loss_total / len(loader)

def prepare_data():
    print("📂 Loading Metadata...")
    df = pd.read_csv(Config.CSV_FILE)
    rows = []
    for pid, grp in df.groupby('patientId'):
        target = grp['Target'].values[0]
        boxes = []
        if target == 1:
            for _, r in grp.iterrows():
                if not pd.isna(r['x']): boxes.append([r['x'], r['y'], r['width'], r['height']])
        rows.append({'patientId': pid, 'Target': target, 'boxes': boxes})
    
    full_df = pd.DataFrame(rows)
    pos = full_df[full_df['Target'] == 1]
    neg = full_df[full_df['Target'] == 0].sample(n=int(len(pos)*1.5), random_state=Config.SEED)
    balanced_df = pd.concat([pos, neg]).sample(frac=1, random_state=Config.SEED).reset_index(drop=True)
    
    train_df, val_df = train_test_split(balanced_df, test_size=0.15, random_state=Config.SEED, stratify=balanced_df['Target'])
    print(f"✅ Data Ready: Train={len(train_df)} | Val={len(val_df)}")
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)

def main():
    torch.cuda.empty_cache(); gc.collect()
    
    train_df, val_df = prepare_data()
    train_trans, val_trans = get_transforms()
    
    # --- FIX WORKERS: set num_workers=Config.NUM_WORKERS which is 1 ---
    train_ds = RSNADataset(train_df, Config.TRAIN_DIR, transforms_set=train_trans)
    val_ds = RSNADataset(val_df, Config.TRAIN_DIR, transforms_set=val_trans)
    
    train_loader = DataLoader(train_ds, batch_size=Config.REAL_BATCH_SIZE, shuffle=True, 
                              num_workers=Config.NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=Config.REAL_BATCH_SIZE, shuffle=False, 
                            num_workers=Config.NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)
    
    print("🚀 Initializing Optimized Dual-Stream Model...")
    model = get_model()
    model.to(Config.DEVICE)
    
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=Config.NUM_EPOCHS, eta_min=Config.MIN_LR
    )
    
    scaler = torch.amp.GradScaler('cuda', enabled=Config.USE_AMP)
    ACCUM = max(1, Config.BATCH_SIZE // Config.REAL_BATCH_SIZE)
    print(f"⚙️ Config: ImgSize={Config.IMG_SIZE} | Accum={ACCUM} | Workers={Config.NUM_WORKERS}")
    
    BEST_MAP = 0.0
    BEST_PATH = "rsna_dual_stream_fpn_best.pth"
    
    for epoch in range(Config.NUM_EPOCHS):
        loss = train_one_epoch(model, optimizer, train_loader, scaler, ACCUM, epoch)
        
        if epoch > 0: scheduler.step()
            
        print(f"Epoch {epoch+1}/{Config.NUM_EPOCHS} | Loss: {loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        mean_ap = get_rsna_score(model, val_loader, Config.DEVICE)
        
        if mean_ap > BEST_MAP:
            print(f"🔥 IMPROVED! Saving model ({BEST_MAP:.4f} -> {mean_ap:.4f})")
            BEST_MAP = mean_ap
            torch.save(model.state_dict(), BEST_PATH)
        else:
            print(f"Current mAP: {mean_ap:.4f} (Best: {BEST_MAP:.4f})")
            
    print(f"\n🏆 TRAINING FINISHED. Best mAP: {BEST_MAP:.4f}")

if __name__ == '__main__':
    main()

📂 Loading Metadata...
✅ Data Ready: Train=12775 | Val=2255
🚀 Initializing Optimized Dual-Stream Model...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 177MB/s] 


⚙️ Config: ImgSize=512 | Accum=2 | Workers=1


Train Ep 1:   0%|          | 0/6388 [00:00<?, ?it/s]

Epoch 1/15 | Loss: 0.1822 | LR: 0.000020
⏳ Running Evaluation...


...

📊 EVAL: AUC = 0.8720 | mAP@0.4-0.75 = 0.2943
Current mAP: 0.2943 (Best: 0.3347)


Train Ep 15:   0%|          | 0/6388 [00:00<?, ?it/s]

Epoch 15/15 | Loss: 0.1111 | LR: 0.000000
⏳ Running Evaluation...


Eval:   0%|          | 0/1128 [00:00<?, ?it/s]

📊 EVAL: AUC = 0.8737 | mAP@0.4-0.75 = 0.2939
Current mAP: 0.2939 (Best: 0.3347)

🏆 TRAINING FINISHED. Best mAP: 0.3347


In [2]:
# ==========================================
# INFERENCE CODE
# ==========================================
import glob

# Định nghĩa lại Dataset cho Test (khớp với Train)
class RSNATestDataset(Dataset):
    def __init__(self, test_dir):
        self.image_paths = glob.glob(os.path.join(test_dir, "*.png"))
        if len(self.image_paths) == 0:
             self.image_paths = glob.glob(os.path.join(test_dir, "*.jpg"))
        
        self.resize = T.Resize((Config.IMG_SIZE, Config.IMG_SIZE), antialias=True)
        
        self.s1_trans = T.Compose([
            T.ToDtype(torch.float32, scale=True),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        self.s2_trans = T.Compose([
            T.RandomEqualize(p=1.0),
            T.ToDtype(torch.float32, scale=True),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image_id = os.path.basename(path).split('.')[0]
        
        img = cv2.imread(path)
        if img is None: img = np.zeros((Config.IMG_SIZE, Config.IMG_SIZE, 3), dtype=np.uint8)
        else: img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
        original_h, original_w = img.shape[:2]
        img_t = torch.from_numpy(img).permute(2, 0, 1) 
        img_resized = self.resize(img_t)
        
        s1 = self.s1_trans(img_resized)
        s2 = self.s2_trans(img_resized)
        combined_img = torch.cat([s1, s2], dim=0) 
        
        return combined_img, image_id, original_h, original_w

def generate_submission(weights_path, test_dir, output_file='submission.csv', conf_thresh=0.25):
    if not os.path.exists(weights_path):
        print(f"❌ Weights not found: {weights_path}")
        return

    test_ds = RSNATestDataset(test_dir)
    test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=Config.NUM_WORKERS)
    
    print(f"🚀 Loading model from {weights_path}...")
    model = get_model()
    
    checkpoint = torch.load(weights_path, map_location=Config.DEVICE)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    else:
        state_dict = checkpoint
        
    if list(state_dict.keys())[0].startswith('module.'):
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
            
    model.load_state_dict(state_dict)
    model.to(Config.DEVICE)
    model.eval()

    submission_results = []
    print("⏳ Running Inference...")
    
    with torch.no_grad():
        for images, image_ids, orig_hs, orig_ws in tqdm(test_loader):
            images = images.to(Config.DEVICE)
            outputs = model(images)
            
            for i, output in enumerate(outputs):
                img_id = image_ids[i]
                scale_w = orig_ws[i].item() / Config.IMG_SIZE
                scale_h = orig_hs[i].item() / Config.IMG_SIZE
                
                prediction_strings = []
                boxes = output['boxes'].cpu().numpy()
                scores = output['scores'].cpu().numpy()
                
                for box, score in zip(boxes, scores):
                    if score > conf_thresh:
                        x1, y1, x2, y2 = box
                        x = x1 * scale_w
                        y = y1 * scale_h
                        w = (x2 - x1) * scale_w
                        h = (y2 - y1) * scale_h
                        prediction_strings.append(f"{score:.4f} {x:.1f} {y:.1f} {w:.1f} {h:.1f}")
                
                submission_results.append({'patientId': img_id, 'PredictionString': " ".join(prediction_strings)})

    df = pd.DataFrame(submission_results)
    df.to_csv(output_file, index=False)
    print(f"✅ Submission saved to {output_file}")

if __name__ == '__main__':
    # Chạy hàm này khi muốn tạo submission
    generate_submission("rsna_dual_stream_fpn_best.pth", '/kaggle/input/rsna-pneumonia-processed-dataset/Test')

🚀 Loading model from rsna_dual_stream_fpn_best.pth...
⏳ Running Inference...


  0%|          | 0/375 [00:00<?, ?it/s]

✅ Submission saved to submission.csv
